# JS15 Praktikum 1 - Klasifikasi Day vs Night

Implementasi pipeline klasifikasi gambar sesuai GitBook: preprocessing, fitur HOG, standardisasi, neural network, evaluasi, dan ekspor artefak model.

In [10]:
import pickle
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from skimage.feature import hog
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

DATASET_ROOT = Path("JS15/images")
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path("images")
    if not DATASET_ROOT.exists():
        raise FileNotFoundError("Folder JS15/images tidak ditemukan.")

    print(f"Dataset: {DATASET_ROOT.resolve()}")
    tensorflow_available = False
    try:
        import tensorflow as tf
        from tensorflow.keras import layers, models
        tensorflow_available = True
        print(f"TensorFlow tersedia: {tf.__version__}")
    except (ImportError, ModuleNotFoundError):
        print("TensorFlow tidak tersedia; menggunakan fallback MLPClassifier.")

Dataset: C:\ListPc\244107020089_MachineLearning\JS15\images
TensorFlow tidak tersedia; menggunakan fallback MLPClassifier.


## Langkah 1-3: Load dan preprocessing dataset

In [11]:
class_names = ["night", "day"]

def load_dataset(directory):
    images, labels = [], []
    for label in class_names:
        for path in sorted((directory / label).glob("*.jpg")):
            image = cv2.imread(str(path))
            if image is None:
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (256, 256))
            images.append(image)
            labels.append(class_names.index(label))
    return images, np.asarray(labels, dtype=np.int64)

X_train_img, y_train_all = load_dataset(DATASET_ROOT / "training")
X_test_img, y_test = load_dataset(DATASET_ROOT / "test")
print(f"Jumlah data training: {len(X_train_img)}")
print(f"Jumlah data testing: {len(X_test_img)}")

Jumlah data training: 240
Jumlah data testing: 160


## Langkah 4-5: Ekstraksi fitur HOG dan normalisasi

In [12]:
def extract_hog(images):
    return np.asarray([
        hog(
            cv2.cvtColor(image, cv2.COLOR_RGB2GRAY),
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            feature_vector=True,
        )
        for image in images
    ])

X_train_feat = extract_hog(X_train_img)
X_test_feat = extract_hog(X_test_img)
scaler = StandardScaler()
X_train_scaled_all = scaler.fit_transform(X_train_feat)
X_test_scaled = scaler.transform(X_test_feat)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_scaled_all, y_train_all, test_size=0.2, random_state=42, stratify=y_train_all
)
print(f"Ukuran fitur HOG: {X_train_feat.shape}")
print(f"Train: {X_train.shape}, validation: {X_val.shape}, test: {X_test_scaled.shape}")

Ukuran fitur HOG: (240, 34596)
Train: (192, 34596), validation: (48, 34596), test: (160, 34596)


## Langkah 6-9: Membangun dan melatih neural network

In [13]:
start = time.perf_counter()
if tensorflow_available:
    model = models.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(128, activation="relu"),
        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(
        X_train, y_train, epochs=20, batch_size=32,
        validation_data=(X_val, y_val), verbose=0,
    )
    test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
    y_pred_prob = model.predict(X_test_scaled, verbose=0).ravel()
    history_values = history.history
else:
    model = MLPClassifier(
        hidden_layer_sizes=(128, 64), max_iter=200, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=10, random_state=42,
    )
    model.fit(X_train, y_train)
    y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
    test_accuracy = model.score(X_test_scaled, y_test)
    test_loss = model.loss_
    history_values = {"accuracy": model.loss_curve_, "val_accuracy": []}
training_time = time.perf_counter() - start
y_pred = (y_pred_prob >= 0.5).astype(np.int64)
print(f"Akurasi test: {test_accuracy:.4f}")
print(f"Loss akhir: {test_loss:.6f}")
print(f"Waktu training: {training_time:.2f} detik")

Akurasi test: 0.8250
Loss akhir: 0.000158
Waktu training: 2.47 detik


## Langkah 10-11: Evaluasi model

In [14]:
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

if tensorflow_available:
    plt.plot(history_values["accuracy"], label="Train accuracy")
    plt.plot(history_values["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

              precision    recall  f1-score   support

       night       0.86      0.78      0.82        80
         day       0.80      0.88      0.83        80

    accuracy                           0.82       160
   macro avg       0.83      0.82      0.82       160
weighted avg       0.83      0.82      0.82       160

Confusion matrix:
[[62 18]
 [10 70]]


## Langkah 12: Ekspor artefak deployment

In [15]:
MODEL_PATH = Path("day_night_model.keras" if tensorflow_available else "day_night_model.pkl")
SCALER_PATH = Path("scaler.pkl")
if tensorflow_available:
    model.save(MODEL_PATH)
else:
    with MODEL_PATH.open("wb") as file:
        pickle.dump(model, file)
with SCALER_PATH.open("wb") as file:
    pickle.dump(scaler, file)
print(f"Model disimpan: {MODEL_PATH}")
print(f"Scaler disimpan: {SCALER_PATH}")

Model disimpan: day_night_model.pkl
Scaler disimpan: scaler.pkl
